# Credit Card Fraud Detection ด้วย PySpark
Notebook นี้ถูกจัดใหม่ให้เป็น workflow สำหรับสร้างโมเดล fraud detection ที่จำเป็นจริง: setup, load/join, data quality, cleaning, feature engineering, temporal split, baseline, class imbalance handling, tuning, final test evaluation และข้อเสนอแนะสำหรับ production/iteration ถัดไป


## 1. Setup Environment
Import library ที่ใช้จริง ตั้งค่า Spark session, project path, checkpoint helper และ metric helper สำหรับประเมินโมเดล binary classification


In [199]:
# คำอธิบาย: นำเข้าไลบรารี ตั้งค่า path/checkpoint และสร้าง helper สำหรับ metric evaluation
import os
import sys
from pathlib import Path
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType, IntegerType
from pyspark.sql.window import Window
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = Path("/Users/tpoot/Desktop/2_big_data/project")
DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"
CHECKPOINT_DIR = PROJECT_DIR / "data" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
def checkpoint_path(name):
    return str(CHECKPOINT_DIR / name)
def checkpoint_exists(name):
    path = CHECKPOINT_DIR / name
    return path.exists() and ((path / "_SUCCESS").exists() or any(path.glob("*.parquet")))
def write_checkpoint(df, name, overwrite=True):
    path = checkpoint_path(name)
    mode = "overwrite" if overwrite else "errorifexists"
    print(f"Writing checkpoint: {path}")
    df.write.mode(mode).parquet(path)
    return spark.read.parquet(path)
def load_or_create_checkpoint(name, build_fn, overwrite=False):
    path = checkpoint_path(name)
    if checkpoint_exists(name) and not overwrite:
        print(f"Loading checkpoint: {path}")
        return spark.read.parquet(path)
    df = build_fn()
    return write_checkpoint(df, name, overwrite=True)
java_home = os.environ.get("JAVA_HOME")
if not java_home or not Path(java_home).exists():
    for candidate in (
        "/opt/homebrew/Cellar/openjdk@17/17.0.16/libexec/openjdk.jdk/Contents/Home",
        "/Library/Java/JavaVirtualMachines/zulu-17.jdk/Contents/Home",
    ):
        if Path(candidate).exists():
            os.environ["JAVA_HOME"] = candidate
            break
def print_binary_metrics(pred_df, label_col="is_fraud_idx", prediction_col="prediction"):
    counts = {
        (float(row[label_col]), float(row[prediction_col])): row["count"]
        for row in pred_df.groupBy(label_col, prediction_col).count().collect()
    }
    tp = counts.get((1.0, 1.0), 0)
    tn = counts.get((0.0, 0.0), 0)
    fp = counts.get((0.0, 1.0), 0)
    fn = counts.get((1.0, 0.0), 0)
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    print("Confusion Matrix:")
    print([[tn, fp], [fn, tp]])
    print("-" * 30)
    print(f"Accuracy:  {accuracy*100:.2f}")
    print(f"Precision: {precision*100:.2f}")
    print(f"Recall:    {recall*100:.2f}")
    print(f"F1-Score:  {f1*100:.2f}")
    print("-" * 30)
def apply_probability_threshold(pred_df, threshold, probability_col="probability", prediction_col="prediction"):
    return pred_df.withColumn(
        prediction_col,
        F.when(vector_to_array(F.col(probability_col))[1] >= threshold, 1.0).otherwise(0.0),
    )
def threshold_search_f1(pred_df, thresholds, label_col="is_fraud_idx", probability_col="probability"):
    threshold_values = sorted({round(float(t), 6) for t in thresholds})
    if not threshold_values:
        raise ValueError("thresholds must not be empty")
    threshold_array = F.array(*[F.lit(t) for t in threshold_values])
    threshold_df = spark.range(len(threshold_values)).select(F.element_at(threshold_array, (F.col("id") + 1).cast("int")).alias("threshold"))
    scored = (
        pred_df.select(F.col(label_col), vector_to_array(F.col(probability_col))[1].alias("fraud_probability"))
        .crossJoin(F.broadcast(threshold_df))
        .withColumn("threshold_prediction", F.when(F.col("fraud_probability") >= F.col("threshold"), 1.0).otherwise(0.0))
    )
    count_rows = scored.groupBy("threshold", label_col, "threshold_prediction").count().collect()
    counts = {
        (round(float(row["threshold"]), 6), float(row[label_col]), float(row["threshold_prediction"])): row["count"]
        for row in count_rows
    }
    rows = []
    for threshold in threshold_values:
        key_threshold = round(threshold, 6)
        tp = counts.get((key_threshold, 1.0, 1.0), 0)
        fp = counts.get((key_threshold, 0.0, 1.0), 0)
        fn = counts.get((key_threshold, 1.0, 0.0), 0)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append((threshold, precision, recall, f1, tp, fp, fn))
    return pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1", "tp", "fp", "fn"]).sort_values("f1", ascending=False)
def find_best_threshold_f1(pred_df, coarse_step=0.01, fine_window=0.03, fine_step=0.001, label_col="is_fraud_idx", probability_col="probability"):
    coarse_thresholds = [round(x * coarse_step, 6) for x in range(1, int(1 / coarse_step))]
    coarse_results = threshold_search_f1(pred_df, coarse_thresholds, label_col=label_col, probability_col=probability_col)
    coarse_best = float(coarse_results.iloc[0]["threshold"])
    fine_start = max(0.001, coarse_best - fine_window)
    fine_end = min(0.999, coarse_best + fine_window)
    fine_count = int(round((fine_end - fine_start) / fine_step)) + 1
    fine_thresholds = [round(fine_start + i * fine_step, 6) for i in range(fine_count)]
    fine_results = threshold_search_f1(pred_df, fine_thresholds, label_col=label_col, probability_col=probability_col)
    return fine_results, coarse_results
def evaluate_binary_predictions(pred_df, threshold=None, label_col="is_fraud_idx"):
    eval_df = apply_probability_threshold(pred_df, threshold) if threshold is not None else pred_df
    evaluator_roc = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")
    evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")
    print(f"Area Under ROC: {evaluator_roc.evaluate(eval_df):.4f}")
    print(f"Area Under PR:  {evaluator_pr.evaluate(eval_df):.4f}")
    if threshold is not None:
        print(f"Threshold:      {threshold:.2f}")
    eval_df.groupBy(label_col).pivot("prediction").count().sort(label_col).show()
    print_binary_metrics(eval_df, label_col=label_col)
    return eval_df


In [200]:
# คำอธิบาย: เคลียร์ Spark session เก่าที่อาจตายค้างอยู่ แล้วสร้าง SparkSession ใหม่สำหรับประมวลผลข้อมูลขนาดใหญ่บนเครื่อง local
temp_path = os.path.expanduser('~/spark-temp')
os.makedirs(temp_path, exist_ok=True)
# If a previous Spark JVM died, SparkSession.builder.getOrCreate() may try to reuse
# a broken gateway and fail with ConnectionRefusedError. Clear old handles first.
try:
    spark.stop()
except Exception:
    pass
try:
    from pyspark import SparkContext
    SparkContext._active_spark_context = None
except Exception:
    pass
spark = (
    SparkSession.builder
    .appName("BigDataComputing")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "3g")
    .config("spark.sql.shuffle.partitions", "80")
    .config("spark.memory.fraction", "0.7")
    .config("spark.local.dir", temp_path)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark Session Created:")
print(spark)
print(f"Project dir: {PROJECT_DIR}")


Spark Session Created:
Project dir: /Users/tpoot/Desktop/2_big_data/project


26/04/25 15:54:30 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


## 2. Load Raw Data
โหลด transactions, cards และ users จาก CSV พร้อมสร้าง MCC/category fallback เพื่อให้ workflow รันได้แม้ไม่มี lookup table แยก


In [201]:
# คำอธิบาย: กำหนด path ของ raw CSV ตรวจสอบไฟล์ที่จำเป็น และโหลด transactions/cards/users/MCC mapping เข้า Spark DataFrame
raw_files = {
    "transactions": DATA_RAW_DIR / "credit_card_transactions-ibm_v2.csv",
    "cards": DATA_RAW_DIR / "sd254_cards.csv",
    "users": DATA_RAW_DIR / "sd254_users.csv",
}
missing = [str(path) for path in raw_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing source CSV files: " + ", ".join(missing))
# Python 3.15 currently lacks reliable wheels for pyarrow on this machine, so
# read CSV directly instead of requiring Parquet support.
transaction = spark.read.csv(str(raw_files["transactions"]), header=True, inferSchema=True)
card = spark.read.csv(str(raw_files["cards"]), header=True, inferSchema=True)
user = spark.read.csv(str(raw_files["users"]), header=True, inferSchema=True)
# The provided project data does not include an MCC lookup table. Keep the
# downstream join/plots working by using MCC itself as the category label.
MCC_list = transaction.select("MCC").distinct().withColumn("Category Name", F.col("MCC").cast("string"))
print("Data loaded from CSV and ready.")


Data loaded from CSV and ready.


## 3. Merge Modeling Table
แปลงเวลา สร้าง user id สำหรับ join และรวม transactions เข้ากับ card/user/category ให้เป็นตารางหลักสำหรับ cleaning, feature engineering และ modeling


In [202]:
# คำอธิบาย: แปลง column เวลาเดิมให้เป็นรูปแบบ HH:mm:ss เพื่อใช้งานต่อกับ timestamp และ feature เวลา
transaction = transaction.withColumn(
    "time",
    F.date_format(F.to_timestamp(F.col("Time"), "HH:mm"), "HH:mm:ss")
)
# transaction = transaction.drop('Time')


In [203]:
# คำอธิบาย: สร้าง User id จากลำดับแถวของ users และ cache ตาราง user ขนาดเล็กเพื่อใช้ join ซ้ำ
# Add the user id from the original users.csv row order without Python RDD/createDataFrame serialization.
# coalesce(1) keeps this small table in one partition so monotonically_increasing_id is 0..1999.
user = user.coalesce(1).withColumn("User", F.monotonically_increasing_id().cast(IntegerType())).cache()
user.count()


2000

In [204]:
# คำอธิบาย: สร้าง merge_df จาก transaction, MCC, card และ user พร้อมเขียน/โหลด Parquet checkpoint หลัง join หลัก
def build_merge_base():
    card_cols = [col(f"c.`{c}`") for c in card.columns if c not in ["User", "CARD INDEX"]]
    user_cols = [col(f"u.`{c}`") for c in user.columns if c not in ["User"]]
    merged = (
        transaction.alias("t")
        .join(MCC_list.alias("m"), col("t.MCC") == col("m.MCC"), "left")
        .join(card.alias("c"), (col("t.Card") == col("c.`CARD INDEX`")) & (col("t.User") == col("c.User")), "left")
        .join(user.alias("u"), col("t.User") == col("u.User"), "left")
        .select(
            col("t.*"),
            col("m.Category Name"),
            *card_cols,
            *user_cols
        )
    )
    return merged.toDF(*[c.lower().replace(' ', '_') for c in merged.columns])
merge_df = load_or_create_checkpoint("01_merge_base", build_merge_base)
merge_df.printSchema()


Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/01_merge_base
root
 |-- user: integer (nullable = true)
 |-- card: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_name: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors?: string (nullable = true)
 |-- is_fraud?: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- cards_issued: integer (nullable

## 4. Data Quality และ Cleaning
ตรวจ missing values, แปลง field เงินจาก string เป็น numeric และดู class distribution เพื่อยืนยันปัญหา class imbalance ก่อน train โมเดล


In [205]:
# คำอธิบาย: นิยามฟังก์ชันตรวจ missing values และสรุปจำนวน/เปอร์เซ็นต์ค่าว่างราย column
def check_missing(df):
    total_rows = df.count()
    if total_rows == 0:
        return "DataFrame is empty."
    expressions = []
    for col_name, data_type in df.dtypes:
        # for Double, Float
        if data_type in ['double', 'float']:
            expressions.append(
                F.count(F.when(F.col(col_name).isNull() | F.isnan(col_name), col_name)).alias(col_name)
            )
        else:
            # for Int, Long, String, Timestamp
            expressions.append(
                F.count(F.when(F.col(col_name).isNull(), col_name)).alias(col_name)
            )
    null_counts_dict = df.select(*expressions).collect()[0].asDict()
    # transform to pandas and calculate %
    summary = pd.DataFrame.from_dict(null_counts_dict, orient='index', columns=['null_count'])
    summary['percentage'] = (summary['null_count'] / total_rows) * 100
    return summary.sort_values('null_count', ascending=False)


In [206]:
# คำอธิบาย: เรียกใช้ฟังก์ชันตรวจ missing values บน merge_df เพื่อดูปัญหาคุณภาพข้อมูล
result = check_missing(merge_df)
print(result)


                             null_count  percentage
errors?                        23998469   98.407215
apartment                      17762699   72.837052
zip                             2878135   11.801972
merchant_state                  2720821   11.156896
user                                  0    0.000000
card_on_dark_web                      0    0.000000
person                                0    0.000000
current_age                           0    0.000000
retirement_age                        0    0.000000
birth_year                            0    0.000000
birth_month                           0    0.000000
gender                                0    0.000000
address                               0    0.000000
city                                  0    0.000000
acct_open_date                        0    0.000000
state                                 0    0.000000
zipcode                               0    0.000000
latitude                              0    0.000000
longitude   

In [207]:
# คำอธิบาย: แปลง column เงินจาก string ที่มีเครื่องหมาย $ เป็น numeric/decimal และบันทึก checkpoint หลัง cleaning
def build_cleaned_merge():
    return (
        merge_df
        .withColumn('amount', F.regexp_replace('amount', r'\$', '').cast(DecimalType(10, 2)))
        .withColumn('per_capita_income_-_zipcode', F.regexp_replace('per_capita_income_-_zipcode', r'\$', '').cast(DecimalType(10, 2)))
        .withColumn('yearly_income_-_person', F.regexp_replace('yearly_income_-_person', r'\$', '').cast(DecimalType(10, 2)))
        .withColumn('total_debt', F.regexp_replace('total_debt', r'\$', '').cast(DecimalType(10, 2)))
        .withColumn('credit_limit', F.regexp_replace('credit_limit', r'\$', '').cast(DecimalType(10, 2)))
    )
merge_df = load_or_create_checkpoint("02_cleaned_merge", build_cleaned_merge)
merge_df.printSchema()


Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/02_cleaned_merge
root
 |-- user: integer (nullable = true)
 |-- card: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_name: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors?: string (nullable = true)
 |-- is_fraud?: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- cards_issued: integer

In [208]:
# คำอธิบาย: คำนวณจำนวนและเปอร์เซ็นต์ fraud/non-fraud เพื่อดู class distribution
total_count = merge_df.count()
merge_df_fraud = (
    merge_df.groupBy('is_fraud?').count()
    .withColumnRenamed('count', 'transaction_count')
    .withColumn('% transaction', F.round((F.col('transaction_count') / total_count * 100), 2))
)
merge_df_fraud.show()


+---------+-----------------+-------------+
|is_fraud?|transaction_count|% transaction|
+---------+-----------------+-------------+
|       No|         24357143|        99.88|
|      Yes|            29757|         0.12|
+---------+-----------------+-------------+



## 5. Feature Engineering
สร้าง feature เวลาและวันที่ เช่น hour, transaction date, account age และ months to expiry จากนั้นสร้าง label numeric สำหรับ pipeline


In [209]:
# คำอธิบาย: แตก hour และ minute ออกจากเวลา transaction เพื่อสร้าง feature เวลา
merge_df = (
    merge_df
    .withColumn('hour', F.substring('time', 1, 2).cast(IntegerType()))
    .withColumn('minute', F.substring('time', 4, 2).cast(IntegerType()))
)


In [210]:
# คำอธิบาย: สร้าง behavioral features แบบลด leakage โดยใช้เฉพาะประวัติก่อนหน้า transaction นั้น
merge_df = merge_df.withColumn(
    "transaction_dt",
    F.to_date(F.concat_ws('-', F.col('year'), F.col('month'), F.col('day')), 'yyyy-M-d')
)
merge_df = merge_df.withColumn("open_dt", F.to_date(F.col("acct_open_date"), "MM/yyyy"))
merge_df = merge_df.withColumn("expire_dt", F.to_date(F.col("expires"), "MM/yyyy"))
merge_df = merge_df.withColumn("account_age_months", F.months_between(F.col("transaction_dt"), F.col("open_dt")))
merge_df = merge_df.withColumn("months_to_expiry", F.months_between(F.col("expire_dt"), F.col("transaction_dt")))
merge_df = merge_df.withColumn("expire_year", F.substring(F.col("expires"), -4, 4).cast("int"))
merge_df = merge_df.withColumn("log_amount", F.log1p(F.greatest(F.col("amount").cast("double"), F.lit(0.0))))
merge_df = merge_df.withColumn(
    "full_timestamp",
    F.to_timestamp(F.concat_ws(' ', F.concat_ws('-', F.col('year'), F.col('month'), F.col('day')), F.col('time')), 'yyyy-M-d HH:mm:ss')
)
user_card_time_window = Window.partitionBy('user', 'card').orderBy('full_timestamp')
user_time_window = Window.partitionBy('user').orderBy('full_timestamp')
user_history_window = user_time_window.rowsBetween(Window.unboundedPreceding, -1)
user_day_history_window = Window.partitionBy('user', 'transaction_dt').orderBy('full_timestamp').rowsBetween(Window.unboundedPreceding, -1)
merchant_day_history_window = Window.partitionBy('merchant_name', 'transaction_dt').orderBy('full_timestamp').rowsBetween(Window.unboundedPreceding, -1)
merge_df = merge_df.withColumn("prev_full_timestamp", F.lag("full_timestamp").over(user_card_time_window))
merge_df = merge_df.withColumn("prev_user_timestamp", F.lag("full_timestamp").over(user_time_window))
merge_df = merge_df.withColumn("prev_user_city", F.lag("merchant_city").over(user_time_window))
merge_df = merge_df.withColumn("user_avg_amount_before", F.avg(F.col("amount").cast("double")).over(user_history_window))
merge_df = merge_df.withColumn("user_freq_day_before", F.count("*").over(user_day_history_window).cast(IntegerType()))
merge_df = merge_df.withColumn("merchant_freq_day_before", F.count("*").over(merchant_day_history_window).cast(IntegerType()))
merge_df = merge_df.withColumn(
    "time_since_prev_txn_minutes",
    F.coalesce((F.unix_timestamp("full_timestamp") - F.unix_timestamp("prev_full_timestamp")) / 60.0, F.lit(999999.0))
)
merge_df = merge_df.withColumn(
    "time_since_prev_user_txn_hours",
    F.coalesce((F.unix_timestamp("full_timestamp") - F.unix_timestamp("prev_user_timestamp")) / 3600.0, F.lit(999999.0))
)
merge_df = merge_df.withColumn("is_first_user_card_txn", F.when(F.col("prev_full_timestamp").isNull(), 1).otherwise(0).cast(IntegerType()))
merge_df = merge_df.withColumn("city_change_24h", F.when((F.col("prev_user_city").isNotNull()) & (F.col("merchant_city") != F.col("prev_user_city")) & (F.col("time_since_prev_user_txn_hours") < 24), 1).otherwise(0).cast(IntegerType()))
merge_df = merge_df.withColumn("user_avg_amount_before", F.coalesce(F.col("user_avg_amount_before"), F.col("amount").cast("double")))
merge_df = merge_df.withColumn("amount_deviation_from_user_avg_before", F.col("amount").cast("double") - F.col("user_avg_amount_before"))


In [211]:
# คำอธิบาย: สร้าง feature base เพิ่ม label numeric และบันทึก checkpoint ก่อนขั้น preprocessing/modeling
def build_feature_base():
    return (
        merge_df
        .withColumn("hour", F.col("hour").cast(IntegerType()))
        .withColumn("minute", F.col("minute").cast(IntegerType()))
        .withColumn("time_since_prev_txn_minutes", F.col("time_since_prev_txn_minutes").cast("double"))
        .withColumn("time_since_prev_user_txn_hours", F.col("time_since_prev_user_txn_hours").cast("double"))
        .withColumn("user_freq_day_before", F.col("user_freq_day_before").cast(IntegerType()))
        .withColumn("merchant_freq_day_before", F.col("merchant_freq_day_before").cast(IntegerType()))
        .withColumn("amount_deviation_from_user_avg_before", F.col("amount_deviation_from_user_avg_before").cast("double"))
        .withColumn("user_avg_amount_before", F.col("user_avg_amount_before").cast("double"))
        .withColumn("log_amount", F.col("log_amount").cast("double"))
        .withColumn("is_fraud_idx", F.when(F.col("is_fraud?") == "Yes", 1).otherwise(0).cast(IntegerType()))
    )
merge_df = load_or_create_checkpoint("04_feature_base_v4_theory_past_features", build_feature_base)


Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/04_feature_base_v4_theory_past_features


## 6. Preprocessing Pipeline
กำหนด categorical columns ที่ cardinality ไม่สูงเกินไป และเตรียม StringIndexer ให้ fit เฉพาะบน train set ภายใน Spark ML Pipeline เพื่อเลี่ยง data leakage


In [212]:
# คำอธิบาย: กำหนด categorical columns ที่สำคัญ รวม category_name กลับมา และใช้ maxBins=128 ให้พอสำหรับ cardinality ประมาณ 109 classes
nominal_binary_cols = [
    'has_chip', 'card_on_dark_web', 'gender', 'use_chip',
    'category_name', 'card_brand', 'card_type', 'state'
]
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in nominal_binary_cols
]
df_with_target = merge_df.withColumn(
    "is_fraud_idx",
    F.when(F.col("is_fraud?") == "Yes", 1).otherwise(0).cast("int")
)
df_with_target.select(nominal_binary_cols + ["is_fraud_idx"]).show(5)


+--------+----------------+------+-----------------+-------------+----------+---------------+-----+------------+
|has_chip|card_on_dark_web|gender|         use_chip|category_name|card_brand|      card_type|state|is_fraud_idx|
+--------+----------------+------+-----------------+-------------+----------+---------------+-----+------------+
|     YES|              No|Female|Swipe Transaction|         7542|Mastercard|Debit (Prepaid)|   GA|           0|
|     YES|              No|  Male|Swipe Transaction|         7922|      Visa|         Credit|   CA|           0|
|     YES|              No|Female|Swipe Transaction|         5411|      Amex|         Credit|   CA|           0|
|     YES|              No|  Male|Swipe Transaction|         7922|Mastercard|          Debit|   IL|           0|
|     YES|              No|  Male|Swipe Transaction|         7922|Mastercard|          Debit|   IL|           0|
+--------+----------------+------+-----------------+-------------+----------+---------------+---

## 7. Feature Set
เลือก numeric และ indexed categorical features สำหรับโมเดล โดยตัด identifier, label, raw text/date และ column ที่เสี่ยง leakage ออกจาก training matrix


In [213]:
# คำอธิบาย: กำหนด feature set สุดท้าย โดยตัด identifier/leakage/raw text ออก และเปิดใช้ feature พฤติกรรมที่รู้ได้ ณ เวลาธุรกรรม
# ตัด merchant_name/cvv เพราะเป็น identifier-like, ตัด mcc เพราะซ้ำกับ category_name_idx, ตัด zip เพราะ null เยอะและทำให้ VectorAssembler skip row
exclude = {
    'user', 'card', 'card_number', 'merchant_name', 'mcc', 'cvv', 'zip', 'apartment', 'errors?', 'person', 'birth_year', 'birth_month',
    'merchant_state', 'latitude', 'longitude', 'is_fraud_idx', 'has_chip', 'is_fraud?', 'card_on_dark_web',
    'gender', 'use_chip', 'merchant_city', 'category_name', 'card_brand', 'card_type', 'address', 'city', 'state',
    'open_dt', 'expire_dt', 'acct_open_date', 'expires', 'time', 'minute', 'transaction_dt', 'date_obj', 'date_ts',
    'full_timestamp', 'prev_full_timestamp', 'prev_user_timestamp', 'prev_user_city', 'split_rand'
}
usable_types = {'int', 'bigint', 'double', 'float', 'long', 'decimal(10,2)'}
numeric_features = [c for c, t in df_with_target.dtypes if c not in exclude and t in usable_types]
indexed_categorical_features = [f"{c}_idx" for c in nominal_binary_cols]
final_features = numeric_features + indexed_categorical_features
assembler = VectorAssembler(inputCols=final_features, outputCol="raw_features", handleInvalid="skip")
featureIndexer = VectorIndexer(inputCol="raw_features", outputCol="features", maxCategories=5)
print(f"Total features: {len(final_features)}")
print(final_features)


Total features: 36
['year', 'month', 'day', 'amount', 'cards_issued', 'credit_limit', 'year_pin_last_changed', 'current_age', 'retirement_age', 'zipcode', 'per_capita_income_-_zipcode', 'yearly_income_-_person', 'total_debt', 'fico_score', 'num_credit_cards', 'hour', 'account_age_months', 'months_to_expiry', 'expire_year', 'log_amount', 'user_avg_amount_before', 'user_freq_day_before', 'merchant_freq_day_before', 'time_since_prev_txn_minutes', 'time_since_prev_user_txn_hours', 'is_first_user_card_txn', 'city_change_24h', 'amount_deviation_from_user_avg_before', 'has_chip_idx', 'card_on_dark_web_idx', 'gender_idx', 'use_chip_idx', 'category_name_idx', 'card_brand_idx', 'card_type_idx', 'state_idx']


## 8. Baseline Random Forest
แบ่งข้อมูลตามเวลาเป็น train/validate/test, train baseline Random Forest บน sampled train เพื่อให้ทดลองเร็ว และประเมินบน validation distribution เดิมด้วย ROC-AUC, PR-AUC, confusion matrix, precision, recall และ F1


In [214]:
# คำอธิบาย: ใช้ year+label stratified random split เพื่อให้ทุก split มีทุกช่วงเวลา แต่ยังคง fraud distribution จริง
# เหมาะเมื่อมี distribution shift แถวปี 2014 และต้องการ validation ที่ไม่เพี้ยนจากการตัดช่วงเวลาแข็งเกินไป
def build_model_base():
    return df_with_target.withColumn(
        'date_obj',
        F.to_date(F.concat_ws('-', F.col('year'), F.col('month'), F.col('day')), 'yyyy-M-d')
    ).withColumn('date_ts', F.unix_timestamp('date_obj'))
df_final = load_or_create_checkpoint("05_model_base_v4_theory_stratified", build_model_base)
df_final = df_final.withColumn("split_rand", F.rand(seed=42))
train_data = load_or_create_checkpoint(
    "06_train_year_label_random_v4_theory",
    lambda: df_final.filter(F.col("split_rand") <= 0.70)
)
validate_data = load_or_create_checkpoint(
    "07_validate_year_label_random_v4_theory",
    lambda: df_final.filter((F.col("split_rand") > 0.70) & (F.col("split_rand") <= 0.85))
)
test_data = load_or_create_checkpoint(
    "08_test_year_label_random_v4_theory",
    lambda: df_final.filter(F.col("split_rand") > 0.85)
)
print("Split label distribution:")
train_data.groupBy("is_fraud_idx").count().show()
validate_data.groupBy("is_fraud_idx").count().show()
test_data.groupBy("is_fraud_idx").count().show()
print("Year x label distribution check:")
train_data.groupBy("year", "is_fraud_idx").count().orderBy("year", "is_fraud_idx").show(100)
validate_data.groupBy("year", "is_fraud_idx").count().orderBy("year", "is_fraud_idx").show(100)
test_data.groupBy("year", "is_fraud_idx").count().orderBy("year", "is_fraud_idx").show(100)


Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/05_model_base_v4_theory_stratified
Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/06_train_year_label_random_v4_theory
Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/07_validate_year_label_random_v4_theory
Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/08_test_year_label_random_v4_theory
Split label distribution:
+------------+--------+
|is_fraud_idx|   count|
+------------+--------+
|           1|   21036|
|           0|17053266|
+------------+--------+

+------------+-------+
|is_fraud_idx|  count|
+------------+-------+
|           1|   4324|
|           0|3650454|
+------------+-------+

+------------+-------+
|is_fraud_idx|  count|
+------------+-------+
|           1|   4397|
|           0|3653423|
+------------+-------+

Year x label distribution check:
+----+------------+-------+
|year|is_fraud_idx|  count|
+----+

In [215]:
# คำอธิบาย: random split ไม่มี cutoff ตามเวลา จึงแสดงเฉพาะจำนวน record ในแต่ละ split
print(f"Train rows:    {train_data.count():,}")
print(f"Validate rows: {validate_data.count():,}")
print(f"Test rows:     {test_data.count():,}")


Train rows:    17,074,302
Validate rows: 3,654,778
Test rows:     3,657,820


In [216]:
# คำอธิบาย: ตรวจ class distribution ของ train, validate และ test เพื่อดูว่า fraud มีอยู่ครบทุก split
train_data.groupBy('is_fraud_idx').count().show()
validate_data.groupBy('is_fraud_idx').count().show()
test_data.groupBy('is_fraud_idx').count().show()


+------------+--------+
|is_fraud_idx|   count|
+------------+--------+
|           1|   21036|
|           0|17053266|
+------------+--------+

+------------+-------+
|is_fraud_idx|  count|
+------------+-------+
|           1|   4324|
|           0|3650454|
+------------+-------+

+------------+-------+
|is_fraud_idx|  count|
+------------+-------+
|           1|   4397|
|           0|3653423|
+------------+-------+



In [217]:
# คำอธิบาย: ลดขนาด train เพื่อให้ local training ไหว โดยเก็บ fraud ทั้งหมดและ sample non-fraud 10% เฉพาะ train เท่านั้น
# Validation/test ยังเป็น distribution จริง ห้าม balance เพราะจะทำให้ metric หลอกตัวเอง
def build_train_data_small():
    fraud_train = train_data.filter(F.col("is_fraud_idx") == 1)
    nonfraud_train_sample = train_data.filter(F.col("is_fraud_idx") == 0).sample(False, 0.10, seed=42)
    return fraud_train.unionByName(nonfraud_train_sample)
train_data_small = load_or_create_checkpoint(
    "09_train_nonfraud_10pct_v6_fast",
    build_train_data_small
)
print("Full train distribution:")
train_data.groupBy("is_fraud_idx").count().show()
print("Sampled train distribution:")
train_data_small.groupBy("is_fraud_idx").count().show()


Loading checkpoint: /Users/tpoot/Desktop/2_big_data/project/data/checkpoints/09_train_nonfraud_10pct_v6_fast
Full train distribution:
+------------+--------+
|is_fraud_idx|   count|
+------------+--------+
|           1|   21036|
|           0|17053266|
+------------+--------+

Sampled train distribution:
+------------+-------+
|is_fraud_idx|  count|
+------------+-------+
|           0|1706682|
|           1|  21036|
+------------+-------+



In [218]:
# คำอธิบาย: สร้าง Pipeline รวม preprocessing, feature assembly และ baseline RandomForest โดย fit เฉพาะ train_data
# Use the same fast RF shape as the weighted experiment for comparable results.
rf = RandomForestClassifier(
    labelCol="is_fraud_idx",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    maxBins=128,
    minInstancesPerNode=20,
    seed=42
)
pipeline = Pipeline(stages=indexers + [assembler, featureIndexer, rf])


In [219]:
# คำอธิบาย: train baseline model ด้วย train_data_small และ unpersist cache หลังใช้เสร็จ
# train model
train_data_small.cache()
model = pipeline.fit(train_data_small)
train_data_small.unpersist()


26/04/25 15:56:16 WARN DAGScheduler: Broadcasting large task binary with size 1445.0 KiB
26/04/25 15:56:30 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/04/25 15:56:47 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/04/25 15:57:10 WARN DAGScheduler: Broadcasting large task binary with size 7.9 MiB
26/04/25 15:57:35 WARN DAGScheduler: Broadcasting large task binary with size 1437.0 KiB
26/04/25 15:57:37 WARN DAGScheduler: Broadcasting large task binary with size 13.1 MiB
26/04/25 15:58:10 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


DataFrame[user: int, card: int, year: int, month: int, day: int, time: string, amount: decimal(10,2), use_chip: string, merchant_name: bigint, merchant_city: string, merchant_state: string, zip: double, mcc: int, errors?: string, is_fraud?: string, category_name: string, card_brand: string, card_type: string, card_number: bigint, expires: string, cvv: int, has_chip: string, cards_issued: int, credit_limit: decimal(10,2), acct_open_date: string, year_pin_last_changed: int, card_on_dark_web: string, person: string, current_age: int, retirement_age: int, birth_year: int, birth_month: int, gender: string, address: string, apartment: int, city: string, state: string, zipcode: int, latitude: double, longitude: double, per_capita_income_-_zipcode: decimal(10,2), yearly_income_-_person: decimal(10,2), total_debt: decimal(10,2), fico_score: int, num_credit_cards: int, hour: int, minute: int, transaction_dt: date, open_dt: date, expire_dt: date, account_age_months: double, months_to_expiry: doub

In [220]:
# คำอธิบาย: สร้าง prediction บน validate_data ด้วย baseline model
# predict
pred = model.transform(validate_data)


In [221]:
# คำอธิบาย: ประเมิน baseline model บน validation set ด้วย ROC-AUC, PR-AUC, confusion matrix และ binary metrics
# Baseline evaluation on validation data.
pred = evaluate_binary_predictions(pred)
pred.select("is_fraud_idx", "prediction", "probability").show(5)


26/04/25 15:58:13 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB


Area Under ROC: 0.9364


26/04/25 15:58:55 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 15:59:28 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB


Area Under PR:  0.2347


26/04/25 15:59:51 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 15:59:51 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 15:59:51 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 15:59:51 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/04/25 16:00:08 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 16:00:08 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 16:00:08 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB


+------------+-------+---+
|is_fraud_idx|    0.0|1.0|
+------------+-------+---+
|           0|3650315|139|
|           1|   3725|599|
+------------+-------+---+



26/04/25 16:00:24 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/04/25 16:00:24 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB


Confusion Matrix:
[[3650315, 139], [3725, 599]]
------------------------------
Accuracy:  99.89
Precision: 81.17
Recall:    13.85
F1-Score:  23.67
------------------------------
+------------+----------+--------------------+
|is_fraud_idx|prediction|         probability|
+------------+----------+--------------------+
|           0|       0.0|[0.99464841783818...|
|           0|       0.0|[0.99665031074704...|
|           0|       0.0|[0.99011834574093...|
|           0|       0.0|[0.99696027595893...|
|           0|       0.0|[0.99299318466706...|
+------------+----------+--------------------+
only showing top 5 rows



In [222]:
# คำอธิบาย: ดึง feature importance จาก RandomForest แล้วเรียงลำดับ feature ที่สำคัญที่สุด
rf_model = model.stages[-1]
importances = rf_model.featureImportances
sorted_features = sorted(zip(final_features, importances), 
                         key=lambda x: x[1], 
                         reverse=True)
for feat, imp in sorted_features:
    print(f"{feat:<30} | {imp*100:.2f}")


category_name_idx              | 27.90
use_chip_idx                   | 20.15
log_amount                     | 5.96
amount_deviation_from_user_avg_before | 5.88
time_since_prev_txn_minutes    | 5.52
amount                         | 4.98
city_change_24h                | 4.14
merchant_freq_day_before       | 3.89
card_type_idx                  | 3.25
year                           | 2.76
credit_limit                   | 2.71
hour                           | 1.99
time_since_prev_user_txn_hours | 1.97
per_capita_income_-_zipcode    | 1.64
user_freq_day_before           | 1.42
yearly_income_-_person         | 1.14
state_idx                      | 1.13
user_avg_amount_before         | 1.04
num_credit_cards               | 0.58
months_to_expiry               | 0.46
total_debt                     | 0.27
zipcode                        | 0.23
current_age                    | 0.22
fico_score                     | 0.18
account_age_months             | 0.14
retirement_age                 | 0.12
exp

## 9. Improve Model
เพิ่ม class weight สำหรับ class imbalance, เลือก probability threshold จาก validation set และทำ manual hyperparameter tuning ขนาดเล็กโดยใช้ PR-AUC เป็น metric หลัก


In [223]:
# คำอธิบาย: คำนวณ class weights แบบอ่อนลงด้วย sqrt(inverse frequency) แล้ว train weighted RandomForest
total_count = train_data_small.count()
fraud_count = train_data_small.filter(F.col("is_fraud_idx") == 1).count()
non_fraud_count = total_count - fraud_count
weight_for_fraud = (total_count / fraud_count) ** 0.5 if fraud_count else 1.0
weight_for_non_fraud = (total_count / non_fraud_count) ** 0.5 if non_fraud_count else 1.0
weighted_data = train_data_small.withColumn(
    "class_weight",
    F.when(F.col("is_fraud_idx") == 1, weight_for_fraud).otherwise(weight_for_non_fraud)
)
rf_weighted = RandomForestClassifier(
    labelCol="is_fraud_idx",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    maxBins=128,
    minInstancesPerNode=20,
    weightCol="class_weight",
    seed=42
)
pipeline_weighted = Pipeline(stages=indexers + [assembler, featureIndexer, rf_weighted])
weighted_data.cache()
model_weight = pipeline_weighted.fit(weighted_data)
weighted_data.unpersist()
print(f"Fraud sqrt weight:     {weight_for_fraud:.4f}")
print(f"Non-fraud sqrt weight: {weight_for_non_fraud:.4f}")


26/04/25 16:01:08 WARN DAGScheduler: Broadcasting large task binary with size 1446.3 KiB
26/04/25 16:01:20 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/04/25 16:01:37 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB
26/04/25 16:02:00 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
26/04/25 16:02:29 WARN DAGScheduler: Broadcasting large task binary with size 1472.7 KiB
26/04/25 16:02:32 WARN DAGScheduler: Broadcasting large task binary with size 13.8 MiB
26/04/25 16:03:15 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


Fraud sqrt weight:     9.0626
Non-fraud sqrt weight: 1.0061


In [224]:
# คำอธิบาย: ประเมิน threshold 5 ค่าแบบเบา ๆ บน validation set แล้วเลือก threshold ที่ F1 ดีที่สุด
threshold_grid = [0.20, 0.30, 0.40, 0.50, 0.60]
base_pred_weight = model_weight.transform(validate_data).cache()
threshold_results = threshold_search_f1(base_pred_weight, threshold_grid)
print(threshold_results.to_string(index=False))
best_threshold = float(threshold_results.iloc[0]["threshold"])
best_validation_f1 = float(threshold_results.iloc[0]["f1"])
print(f"Best threshold by validation F1: {best_threshold:.2f}")
print(f"Best validation F1: {best_validation_f1:.4f}")
pred_weight = evaluate_binary_predictions(base_pred_weight, threshold=best_threshold)
base_pred_weight.unpersist()


26/04/25 16:03:21 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/04/25 16:04:30 WARN MemoryStore: Not enough space to cache rdd_613_6 in memory! (computed 382.9 MiB so far)
26/04/25 16:04:30 WARN BlockManager: Persisting block rdd_613_6 to disk instead.
26/04/25 16:04:39 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/04/25 16:04:41 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB


 threshold  precision   recall       f1   tp     fp   fn
       0.6   0.209523 0.404024 0.275944 1747   6591 2577
       0.5   0.137704 0.502313 0.216152 2172  13601 2152
       0.4   0.087105 0.595976 0.151995 2577  27008 1747
       0.3   0.051644 0.669288 0.095888 2894  53144 1430
       0.2   0.027690 0.750000 0.053409 3243 113874 1081
Best threshold by validation F1: 0.60
Best validation F1: 0.2759


26/04/25 16:04:42 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/04/25 16:04:51 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB


Area Under ROC: 0.9545


Area Under PR:  0.1671
Threshold:      0.60


...

+------------+-------+----+
|is_fraud_idx|    0.0| 1.0|
+------------+-------+----+
|           0|3643863|6591|
|           1|   2577|1747|
+------------+-------+----+



26/04/25 16:05:07 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB


Confusion Matrix:
[[3643863, 6591], [2577, 1747]]
------------------------------
Accuracy:  99.75
Precision: 20.95
Recall:    40.40
F1-Score:  27.59
------------------------------


26/04/25 16:05:08 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB


DataFrame[user: int, card: int, year: int, month: int, day: int, time: string, amount: decimal(10,2), use_chip: string, merchant_name: bigint, merchant_city: string, merchant_state: string, zip: double, mcc: int, errors?: string, is_fraud?: string, category_name: string, card_brand: string, card_type: string, card_number: bigint, expires: string, cvv: int, has_chip: string, cards_issued: int, credit_limit: decimal(10,2), acct_open_date: string, year_pin_last_changed: int, card_on_dark_web: string, person: string, current_age: int, retirement_age: int, birth_year: int, birth_month: int, gender: string, address: string, apartment: int, city: string, state: string, zipcode: int, latitude: double, longitude: double, per_capita_income_-_zipcode: decimal(10,2), yearly_income_-_person: decimal(10,2), total_debt: decimal(10,2), fico_score: int, num_credit_cards: int, hour: int, minute: int, transaction_dt: date, open_dt: date, expire_dt: date, account_age_months: double, months_to_expiry: doub

In [225]:
# คำอธิบาย: manual auto-tuning 10 candidates รอบค่าแนะนำหลัก โดยคุม runtime ด้วย train sample 10%
# ค่า base 100 trees / depth 10 / minInstances 20 / maxBins 128 ไม่เยอะเกินไปสำหรับ sample 10%; maxBins 128 จำเป็นเพื่อรองรับ category_name ~109 classes
tuning_candidates = [
    {"numTrees": 60, "maxDepth": 8, "minInstancesPerNode": 20, "maxBins": 128},
    {"numTrees": 80, "maxDepth": 8, "minInstancesPerNode": 20, "maxBins": 128},
    {"numTrees": 100, "maxDepth": 8, "minInstancesPerNode": 20, "maxBins": 128},
    {"numTrees": 100, "maxDepth": 10, "minInstancesPerNode": 20, "maxBins": 128},
    {"numTrees": 120, "maxDepth": 10, "minInstancesPerNode": 20, "maxBins": 128},
    {"numTrees": 100, "maxDepth": 10, "minInstancesPerNode": 40, "maxBins": 128},
    {"numTrees": 120, "maxDepth": 12, "minInstancesPerNode": 40, "maxBins": 128},
    {"numTrees": 140, "maxDepth": 10, "minInstancesPerNode": 40, "maxBins": 128},
    {"numTrees": 160, "maxDepth": 12, "minInstancesPerNode": 60, "maxBins": 128},
    {"numTrees": 200, "maxDepth": 10, "minInstancesPerNode": 60, "maxBins": 128},
]


In [226]:
# คำอธิบาย: ใช้ weighted sampled train ทั้งชุดสำหรับ tuning เพราะ non-fraud ถูกลดเหลือ 10% แล้ว
tuning_data = weighted_data.cache()
print("Tuning distribution:")
tuning_data.groupBy("is_fraud_idx").count().show()


Tuning distribution:


+------------+-------+
|is_fraud_idx|  count|
+------------+-------+
|           0|1706682|
|           1|  21036|
+------------+-------+



In [227]:
# คำอธิบาย: train candidate models 10 ชุดและเลือก best model ด้วย validation PR-AUC พร้อมเก็บ F1 ที่ threshold 0.50 เทียบเบื้องต้น
evaluator_pr = BinaryClassificationEvaluator(labelCol="is_fraud_idx", metricName="areaUnderPR")
evaluator_roc = BinaryClassificationEvaluator(labelCol="is_fraud_idx", metricName="areaUnderROC")
tuning_results = []
best_tuned_model = None
best_tuned_score = -1.0
best_tuned_params = None
for params in tuning_candidates:
    rf_tuned = RandomForestClassifier(
        labelCol="is_fraud_idx",
        featuresCol="features",
        numTrees=params["numTrees"],
        maxDepth=params["maxDepth"],
        minInstancesPerNode=params["minInstancesPerNode"],
        maxBins=params["maxBins"],
        weightCol="class_weight",
        seed=42
    )
    tuned_pipeline = Pipeline(stages=indexers + [assembler, featureIndexer, rf_tuned])
    tuned_model = tuned_pipeline.fit(tuning_data)
    tuned_pred = tuned_model.transform(validate_data).cache()
    pr_auc = evaluator_pr.evaluate(tuned_pred)
    roc_auc = evaluator_roc.evaluate(tuned_pred)
    quick_f1 = float(threshold_search_f1(tuned_pred, [0.50]).iloc[0]["f1"])
    tuned_pred.unpersist()
    tuning_results.append({**params, "areaUnderPR": pr_auc, "areaUnderROC": roc_auc, "f1_at_050": quick_f1})
    if pr_auc > best_tuned_score:
        best_tuned_score = pr_auc
        best_tuned_model = tuned_model
        best_tuned_params = params
tuning_data.unpersist()
tuning_results_df = pd.DataFrame(tuning_results).sort_values("areaUnderPR", ascending=False)
print(tuning_results_df.to_string(index=False))
print(f"Best params by validation PR-AUC: {best_tuned_params}")


26/04/25 16:06:13 WARN DAGScheduler: Broadcasting large task binary with size 1679.0 KiB
26/04/25 16:06:24 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/04/25 16:06:41 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/25 16:07:44 WARN MemoryStore: Not enough space to cache rdd_868_9 in memory! (computed 382.9 MiB so far)
26/04/25 16:07:44 WARN BlockManager: Persisting block rdd_868_9 to disk instead.
26/04/25 16:07:47 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/25 16:08:00 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/25 16:08:10 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/04/25 16:08:11 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/04/25 16:09:12 WARN DAGScheduler: Broadcasting large task binary with size 1215.8 KiB
26/04/25 16:09:27 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/25 16:09:46 WARN DAGSc

Py4JJavaError: An error occurred while calling o22151.fit.
: org.apache.spark.SparkException: Job 357 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1253)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1251)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1251)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3087)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:84)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:2973)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1375)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:2973)
	at org.apache.spark.SparkContext.$anonfun$stop$12(SparkContext.scala:2258)
	at org.apache.s

In [ ]:
# คำอธิบาย: แสดง hyperparameters ของ tuned RandomForest ที่เลือกได้
# Inspect the selected tuned Random Forest.
extract_rf = best_tuned_model.stages[-1]
print(f"Best MaxDepth: {extract_rf.getOrDefault('maxDepth')}")
print(f"Best NumTrees: {extract_rf.getOrDefault('numTrees')}")
print(f"Best validation PR-AUC: {best_tuned_score:.4f}")


In [ ]:
# คำอธิบาย: เอา tuned model ไป predict validate_data แล้วหา threshold/F1 ที่ดีที่สุดจาก threshold 5 ค่า
threshold_grid = [0.20, 0.30, 0.40, 0.50, 0.60]
tuned_validation_pred = best_tuned_model.transform(validate_data).cache()
tuned_threshold_results = threshold_search_f1(tuned_validation_pred, threshold_grid)
print(tuned_threshold_results.to_string(index=False))
selected_threshold = float(tuned_threshold_results.iloc[0]["threshold"])
selected_validation_f1 = float(tuned_threshold_results.iloc[0]["f1"])
print(f"Selected tuned-model threshold by validation F1: {selected_threshold:.2f}")
print(f"Selected tuned-model validation F1: {selected_validation_f1:.4f}")
pred_tuned = evaluate_binary_predictions(tuned_validation_pred, threshold=selected_threshold)
pred_tuned.select("is_fraud_idx", "prediction", "probability").show(5)
tuned_validation_pred.unpersist()


In [ ]:
# คำอธิบาย: สรุป F1 บน validation จากโมเดลที่เลือกได้ก่อนนำ threshold ไปใช้กับ final/test
validation_f1_summary = pd.DataFrame([
    {"model": "weighted_rf", "threshold": best_threshold, "validation_f1": best_validation_f1},
    {"model": "tuned_rf", "threshold": selected_threshold, "validation_f1": selected_validation_f1}
]).sort_values("validation_f1", ascending=False)
print(validation_f1_summary.to_string(index=False))


## 10. Final Model
หลังเลือก feature set, class weight, hyperparameters และ threshold จาก validation แล้ว รวม train+validate เพื่อ train final model และประเมินเพียงครั้งเดียวบน untouched test set


In [ ]:
# คำอธิบาย: train final model บน train+validate แบบ sampled เพื่อให้รันเร็ว แล้วประเมินบน untouched test_data distribution จริง
train_validate_data = train_data.unionByName(validate_data)
def build_final_train_sample():
    fraud_rows = train_validate_data.filter(F.col("is_fraud_idx") == 1)
    nonfraud_rows = train_validate_data.filter(F.col("is_fraud_idx") == 0).sample(False, 0.10, seed=42)
    return fraud_rows.unionByName(nonfraud_rows)
final_train_sample = build_final_train_sample()
final_total_count = final_train_sample.count()
final_fraud_count = final_train_sample.filter(F.col("is_fraud_idx") == 1).count()
final_non_fraud_count = final_total_count - final_fraud_count
final_weight_for_fraud = (final_total_count / final_fraud_count) ** 0.5 if final_fraud_count else 1.0
final_weight_for_non_fraud = (final_total_count / final_non_fraud_count) ** 0.5 if final_non_fraud_count else 1.0
train_validate_weighted = final_train_sample.withColumn(
    "class_weight",
    F.when(F.col("is_fraud_idx") == 1, final_weight_for_fraud).otherwise(final_weight_for_non_fraud)
)
selected_params = best_tuned_params if "best_tuned_params" in globals() and best_tuned_params else {
    "numTrees": 100,
    "maxDepth": 10,
    "minInstancesPerNode": 20,
    "maxBins": 128
}
selected_threshold = selected_threshold if "selected_threshold" in globals() else (best_threshold if "best_threshold" in globals() else 0.50)
final_rf = RandomForestClassifier(
    labelCol="is_fraud_idx",
    featuresCol="features",
    numTrees=selected_params["numTrees"],
    maxDepth=selected_params["maxDepth"],
    minInstancesPerNode=selected_params.get("minInstancesPerNode", 20),
    maxBins=selected_params.get("maxBins", 128),
    weightCol="class_weight",
    seed=42
)
final_pipeline = Pipeline(stages=indexers + [assembler, featureIndexer, final_rf])
train_validate_weighted.cache()
final_model = final_pipeline.fit(train_validate_weighted)
train_validate_weighted.unpersist()
final_test_pred = final_model.transform(test_data)
final_test_pred = evaluate_binary_predictions(final_test_pred, threshold=selected_threshold)
final_test_pred.select("is_fraud_idx", "prediction", "probability").show(10)
print(f"Final fraud sqrt weight:     {final_weight_for_fraud:.4f}")
print(f"Final non-fraud sqrt weight: {final_weight_for_non_fraud:.4f}")


In [ ]:
# คำอธิบาย: วิเคราะห์ผลท้ายสุดเพิ่มแบบเบา ๆ ด้วย feature importance และ error buckets บน test prediction
final_rf_model = final_model.stages[-1]
final_importance = pd.DataFrame({"feature": final_features, "importance": final_rf_model.featureImportances.toArray()})
final_importance = final_importance.sort_values("importance", ascending=False)
print("Top 20 final feature importances:")
print(final_importance.head(20).to_string(index=False))
print("False negatives by year:")
final_test_pred.where((F.col("is_fraud_idx") == 1) & (F.col("prediction") == 0.0)).groupBy("year").count().orderBy("year").show(100)
print("False positives by year:")
final_test_pred.where((F.col("is_fraud_idx") == 0) & (F.col("prediction") == 1.0)).groupBy("year").count().orderBy("year").show(100)
print("Fraud recall by year on test:")
final_test_pred.groupBy("year").agg(
    F.sum(F.when(F.col("is_fraud_idx") == 1, 1).otherwise(0)).alias("fraud_count"),
    F.sum(F.when((F.col("is_fraud_idx") == 1) & (F.col("prediction") == 1.0), 1).otherwise(0)).alias("fraud_caught")
).withColumn("recall", F.round(F.col("fraud_caught") / F.col("fraud_count"), 4)).orderBy("year").show(100)


## 11. ข้อเสนอแนะเพิ่มเติมสำหรับการปรับปรุงแบบ Senior Data Scientist
- ตั้ง metric หลักเป็น PR-AUC และ recall@precision target เช่น ต้องการ precision อย่างน้อย 20% แล้ว maximize recall เพราะ fraud เป็น class imbalance และ AUROC อาจดูดีเกินจริง
- เลือก threshold จาก expected cost ไม่ใช่ F1 อย่างเดียว เช่น cost ของ false negative = เงินที่เสียจาก fraud และ cost ของ false positive = ค่า manual review/ประสบการณ์ลูกค้า
- ทำ temporal validation แบบ production simulation: train จากเดือนเก่า validate จากเดือนถัดไป test จากช่วงล่าสุด หลีกเลี่ยง random CV สำหรับธุรกรรมตามเวลา
- เพิ่ม velocity features จากประวัติก่อนหน้าเท่านั้น เช่น จำนวน transaction ของ user/card ใน 5 นาที, 1 ชั่วโมง, 24 ชั่วโมง, ยอดรวมย้อนหลัง และจำนวน merchant/category ที่ไม่ซ้ำ
- เพิ่ม deviation features เช่น amount เทียบกับ median/mean ของ user, category, merchant หรือ card ในอดีต เพื่อจับ transaction ที่ผิดปกติจากพฤติกรรมเดิม
- จัดการ high-cardinality categorical อย่าง `merchant_city`, `address`, `city` ด้วย frequency/count encoding หรือ rare-category grouping จาก train เท่านั้น แทนการให้ StringIndexer สร้างเลขลำดับตรง ๆ
- ตรวจ data leakage ทุก feature โดยถามว่า feature นี้รู้ได้จริง ณ เวลาที่ transaction เกิดไหม ถ้าเป็นข้อมูลหลังเหตุการณ์หรือ label-derived ต้องตัดออก
- ทำ error analysis แยก false negative และ false positive: ดูตาม amount bucket, channel, category, state, hour และ customer segment เพื่อรู้ว่าโมเดลพลาด pattern ไหน
- ลดภาระ training ด้วย candidate grid เล็ก, stratified tuning sample, caching เฉพาะจุด และหลีกเลี่ยง CrossValidator หนัก ๆ บนเครื่อง local
- บันทึก experiment table ทุกครั้ง: feature set, split date, params, threshold, PR-AUC, precision, recall, F1, confusion matrix เพื่อให้เปรียบเทียบโมเดลได้อย่าง reproducible
- หลังได้โมเดลดีแล้วควรทำ calibration check เพราะ probability จาก Random Forest อาจไม่ calibrated ถ้าจะใช้ score เป็น risk probability จริง
- ก่อนนำไปใช้จริงควรทำ monitoring plan เช่น fraud rate drift, score distribution drift, precision/recall จาก delayed labels และ retraining cadence
- ใช้ Parquet checkpoint หลังจุดหนัก เช่น merge, cleaning และ model-ready split เพื่อ restart แล้วโหลดต่อได้ทันทีแทนการรัน CSV join ใหม่ทั้งหมด
- สำหรับรอบทดลองเร็ว ใช้ `train_data_small` โดยเก็บ fraud ทั้งหมดและสุ่ม non-fraud บางส่วน แต่ต้องประเมินบน validation/test distribution เดิมเสมอ
